In [2]:
import numpy as np 
import pandas as pd
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from catboost import CatBoostClassifier
import shap

/home/sunsun/miniconda3/envs/prescriptive_datascience/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
corn_df = pd.read_csv('data/CornCostReturn.csv')
corn_df.head()

,Commodity,Category,Item,Units,Size,Region,Country,Year,Value,Survey base year
0,Corn,Gross value of production,"Primary product, grain",dollars per planted acre,No specific size,Eastern Uplands,United States,2025,757.13,Base survey of 2021
1,Corn,Supporting information,Price,dollars per bushel at harvest,No specific size,Prairie Gateway,United States,2025,4.00,Base survey of 2021
2,Corn,Operating costs,Interest on operating capital,dollars per planted acre,No specific size,Southern Seaboard,United States,2025,10.05,Base survey of 2021
3,Corn,Operating costs,"Total, operating costs",dollars per planted acre,No specific size,Eastern Uplands,United States,2025,476.28,Base survey of 2021
4,Corn,Supporting information,Price,dollars per bushel at harvest,No specific size,Northern Great Plains,United States,2025,3.71,Base survey of 2021


In [4]:
cotton_df = pd.read_csv('data/CottonCostReturn.csv')
cotton_df.head()

,Commodity,Category,Item,Units,Size,Region,Country,Year,Value,Survey base year
0,Cotton,Net value,Value of production less operating costs,dollars per planted acre,No specific size,Prairie Gateway,United States,2025,131.34,Base survey of 2019
1,Cotton,Net value,Value of production less operating costs,dollars per planted acre,No specific size,Mississippi Portal,United States,2025,145.03,Base survey of 2019
2,Cotton,Net value,Value of production less total costs listed,dollars per planted acre,No specific size,Fruitful Rim,United States,2025,103.12,Base survey of 2019
3,Cotton,Allocated overhead,Taxes and insurance,dollars per planted acre,No specific size,Prairie Gateway,United States,2025,4.95,Base survey of 2019
4,Cotton,Allocated overhead,Hired labor,dollars per planted acre,No specific size,Prairie Gateway,United States,2025,24.07,Base survey of 2019


In [6]:
soybean_df = pd.read_csv('data/SoybeanCostReturn.csv')
soybean_df.head()

,Commodity,Category,Item,Units,Size,Region,Country,Year,Value,Survey base year
0,Soybean,Allocated overhead,Hired labor,dollars per planted acre,No specific size,Prairie Gateway,United States,2025,13.80,Base survey of 2023
1,Soybean,Allocated overhead,Taxes and insurance,dollars per planted acre,No specific size,Prairie Gateway,United States,2025,14.24,Base survey of 2023
2,Soybean,Operating costs,Purchased irrigation water,dollars per planted acre,No specific size,Southern Seaboard,United States,2025,0.18,Base survey of 2023
3,Soybean,Supporting information,Enterprise size,planted acres,No specific size,Eastern Uplands,United States,2025,375.00,Base survey of 2023
4,Soybean,Allocated overhead,Hired labor,dollars per planted acre,No specific size,Mississippi Portal,United States,2025,18.57,Base survey of 2023


In [7]:
wheat_df = pd.read_csv('data/WheatCostReturn.csv')
wheat_df.head(15)

,Commodity,Category,Item,Units,Size,Region,Country,Year,Value,Survey base year
0,Wheat,Operating costs,"Total, operating costs",dollars per planted acre,No specific size,Heartland,United States,2025,230.85,Base survey of 2022
1,Wheat,Allocated overhead,General farm overhead,dollars per planted acre,No specific size,Heartland,United States,2025,7.70,Base survey of 2022
2,Wheat,Operating costs,Fertilizer,dollars per planted acre,No specific size,Heartland,United States,2025,103.70,Base survey of 2022
3,Wheat,Operating costs,Seed,dollars per planted acre,No specific size,Northern Crescent,United States,2025,30.21,Base survey of 2022
4,Wheat,Operating costs,Other variable expenses,dollars per planted acre,No specific size,Prairie Gateway,United States,2025,0.34,Base survey of 2022
5,Wheat,Operating costs,"Total, operating costs",dollars per planted acre,No specific size,Basin and Range,United States,2025,201.28,Base survey of 2022
6,Wheat,Operating costs,Interest on operating inputs,dollars per planted acre,No specific size,Prairie Gateway,United States,2025,2.85,Base survey of 2022
7,Wheat,Gross value of production,"Total, gross value of production",dollars per planted acre,No specific size,U.S. total,United States,2025,280.32,Base survey of 2022
8,Wheat,Allocated overhead,Taxes and insurance,dollars per planted acre,No specific size,U.S. total,United States,2025,8.26,Base survey of 2022
9,Wheat,Gross value of production,"Primary product, grain",dollars per planted acre,No specific size,Northern Great Plains,United States,2025,231.55,Base survey of 2022


In [8]:
crop_forecast_df = pd.read_csv('data/cop_forecast.csv')
crop_forecast_df.head(10)

,Commodity,Category,Item,Units,Region,Year,Value
0,Corn,Operating costs,Seed,Dollars per planted acre,U.S. total,2026F,111.06
1,Corn,Operating costs,Fertilizer,Dollars per planted acre,U.S. total,2026F,184.33
2,Corn,Operating costs,Chemicals,Dollars per planted acre,U.S. total,2026F,46.44
3,Corn,Operating costs,Custom operations,Dollars per planted acre,U.S. total,2026F,38.56
4,Corn,Operating costs,"Fuel, lube, and electricity",Dollars per planted acre,U.S. total,2026F,34.19
5,Corn,Operating costs,Repairs,Dollars per planted acre,U.S. total,2026F,50.40
6,Corn,Operating costs,Other variable expenses,Dollars per planted acre,U.S. total,2026F,0.30
7,Corn,Operating costs,Interest on operating capital,Dollars per planted acre,U.S. total,2026F,7.75
8,Corn,Operating costs,"Total, operating costs",Dollars per planted acre,U.S. total,2026F,473.03
9,Corn,Allocated overhead,Hired labor,Dollars per planted acre,U.S. total,2026F,4.78


In [9]:
for df in (corn_df, cotton_df, soybean_df, wheat_df, crop_forecast_df):
    df.info()
    print('\n')

<class 'pandas.DataFrame'>
RangeIndex: 5665 entries, 0 to 5664
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Commodity         5665 non-null   str    
 1   Category          5665 non-null   str    
 2   Item              5665 non-null   str    
 3   Units             5665 non-null   str    
 4   Size              5665 non-null   str    
 5   Region            5665 non-null   str    
 6   Country           5665 non-null   str    
 7   Year              5665 non-null   int64  
 8   Value             5665 non-null   float64
 9   Survey base year  5665 non-null   str    
dtypes: float64(1), int64(1), str(8)
memory usage: 442.7 KB


<class 'pandas.DataFrame'>
RangeIndex: 5550 entries, 0 to 5549
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Commodity         5550 non-null   str    
 1   Category          5550 non-null   st

In [10]:
for df in (corn_df, cotton_df, soybean_df, wheat_df, crop_forecast_df):
    str_cols = df.select_dtypes(include=['object', 'string'])
    for col in str_cols:
        print(f'\n', str_cols[col].value_counts())
    print('----------------------------\n')


 Commodity
Corn    5665
Name: count, dtype: int64

 Category
Operating costs              1890
Allocated overhead           1470
Gross value of production     630
Supporting information        630
Net value                     420
Production practices          415
Costs listed                  210
Name: count, dtype: int64

 Item
Primary product, grain                         210
Price                                          210
Interest on operating capital                  210
Total, operating costs                         210
Total, gross value of production               210
Value of production less total costs listed    210
Hired labor                                    210
Total, allocated overhead                      210
Irrigated                                      210
Fertilizer                                     210
Value of production less operating costs       210
General farm overhead                          210
Taxes and insurance                            210
Ente

In [15]:
df_filtered = corn_df[corn_df['Units'].str.contains('planted acres', case=False, na=False)]
print(df_filtered)

     Commodity                Category             Item          Units  \
22        Corn  Supporting information  Enterprise size  planted acres   
44        Corn  Supporting information  Enterprise size  planted acres   
87        Corn  Supporting information  Enterprise size  planted acres   
98        Corn  Supporting information  Enterprise size  planted acres   
103       Corn  Supporting information  Enterprise size  planted acres   
...        ...                     ...              ...            ...   
3995      Corn  Supporting information  Enterprise size  planted acres   
3996      Corn  Supporting information  Enterprise size  planted acres   
3997      Corn  Supporting information  Enterprise size  planted acres   
3998      Corn  Supporting information  Enterprise size  planted acres   
3999      Corn  Supporting information  Enterprise size  planted acres   

                  Size             Region        Country  Year  Value  \
22    No specific size          Heartl

## Forecasting